In [5]:
import pandas as pd
import numpy as np

df_features_norm = pd.read_csv("features_rl_sp100.csv", index_col=0, parse_dates=True)

df_precos = pd.read_csv("matriz_final_v3_survivorship.csv", index_col=0, parse_dates=True)
df_retornos = df_precos.pct_change().dropna()

indices_comuns = df_features_norm.index.intersection(df_retornos.index)

df_features_norm = df_features_norm.loc[indices_comuns]
df_retornos = df_retornos.loc[indices_comuns]

split_idx = int(len(df_features_norm) * 0.8)

train_features = df_features_norm.iloc[:split_idx]
train_returns = df_retornos.iloc[:split_idx]

test_features = df_features_norm.iloc[split_idx:]
test_returns = df_retornos.iloc[split_idx:]

print(f"Pronto! Dados carregados.")
print(f"Treino: {train_features.shape}")

Pronto! Dados carregados.
Treino: (1001, 1300)


In [6]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces    

class PortfolioEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, features, returns, lambda_risk=0.5, turnover_cost=0.001):
        super(PortfolioEnv, self).__init__()

        self.features = features.values # Matriz de estados
        self.returns = returns.values # Matriz de retornos reais dos ativos
        self.asset_names = returns.columns

        self.n_assets = returns.shape[1]
        self.n_features = features.shape[1]
        self.current_step = 0

        # Hiperparâmetros da Recompensa 
        self.lambda_risk = lambda_risk      # Penalidade por volatilidade
        self.turnover_cost = turnover_cost  # Custo de transação/mudança

        # Espaço de Ação: vetor contínuo de pesos (0 a 1) para cada ativo
        self.action_space = spaces.Box(low=0, high=1, shape=(self.n_assets,), dtype=np.float32)
        
        # Espaço de Observação
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.n_features,), dtype=np.float32)
        
        # Estado inicial dos pesos (começa 100% em caixa ou distribuído igualmente)
        self.last_weights = np.ones(self.n_assets) / self.n_assets

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.last_weights = np.ones(self.n_assets) / self.n_assets
        
        # Retorna a primeira observação e info vazia
        return self.features[self.current_step], {}

    def step(self, action):
        # Normalizar ação para garantir soma = 1 (Alocação de Portfólio) 
        # Usa softmax para garantir que todos sejam positivos e somem 1
        weights = np.exp(action) / np.sum(np.exp(action))
        
        # Pegar o retorno real do dia atual (o que aconteceu no mercado)
        daily_returns = self.returns[self.current_step]
        
        # Calcular retorno do portfólio: soma(pesos * retorno do ativo)
        portfolio_return = np.sum(weights * daily_returns)
        
        # Calcular custo de turnover (penalidade por mudar muito os pesos)
        turnover = np.sum(np.abs(weights - self.last_weights))
        
        # Calcular risco (volatilidade instantânea aproximada pelo retorno quadrático ou janela)
        # Simplificação para recompensa instantânea: Risco ~ (Retorno do portfólio)^2 ou volatilidade recente
        portfolio_risk = portfolio_return ** 2 # Proxy simples de variância local
        
        # Função de recompensa 
        # Reward = Retorno - (lambda * Risco) - (custo * turnover)
        reward = portfolio_return - (self.lambda_risk * portfolio_risk) - (self.turnover_cost * turnover)
        
        # Atualizar estado
        self.last_weights = weights
        self.current_step += 1
        
        # Verificar se acabou os dados
        terminated = self.current_step >= len(self.features) - 1
        truncated = False
        
        # Próximo estado
        next_observation = self.features[self.current_step]
        
        info = {
            'portfolio_return': portfolio_return,
            'turnover': turnover,
            'weights': weights
        }
        
        return next_observation, reward, terminated, truncated, info

    def render(self, mode='human'):
        pass

In [7]:
# Teste simples do ambiente (Aleatório)
env = PortfolioEnv(train_features, train_returns)
obs, _ = env.reset()

done = False
total_reward = 0

print("Rodando simulação aleatória...")
while not done:
    # Agente aleatório: escolhe pesos aleatórios
    action = env.action_space.sample() 
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated

print(f"Simulação concluída! Recompensa acumulada (Aleatória): {total_reward:.4f}")

Rodando simulação aleatória...
Simulação concluída! Recompensa acumulada (Aleatória): 0.2177


In [8]:
from stable_baselines3 import PPO

env = PortfolioEnv(train_features, train_returns)

model = PPO("MlpPolicy", env, verbose=1)

model.learn(total_timesteps=20000)

model.save("robo_investidor")

print(f"DATA DE CORTE: {test_features.index[0]}")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 0.0892   |
| time/              |          |
|    fps             | 483      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1e+03      |
|    ep_rew_mean          | 0.101      |
| time/                   |            |
|    fps                  | 331        |
|    iterations           | 2          |
|    time_elapsed         | 12         |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.19312555 |
|    clip_fraction        | 0.673      |
|    clip_range           | 0.2        |
|    entropy_loss         | -142 